In [1]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="st_paraphrase_minilm_l3_cosine_threshold_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [2]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


---[ TableVault Record ]---
device: mps
---[ TableVault Record ]---



In [5]:
model_name = "sentence-transformers/paraphrase-MiniLM-L3-v2"
threshold = 0.74
batch_size = 128

model = SentenceTransformer(model_name, device=str(device))
print(model_name)
print({"threshold": threshold, "batch_size": batch_size})


---[ TableVault Record ]---


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers/paraphrase-MiniLM-L3-v2
{'threshold': 0.74, 'batch_size': 128}
---[ TableVault Record ]---



In [6]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
---[ TableVault Record ]---



In [7]:
emb1 = model.encode(
    sent1,
    batch_size=batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
emb1_list = emb1.cpu().float().tolist()
emb2 = model.encode(
    sent2,
    batch_size=batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
emb2_list = emb2.cpu().float().tolist()



---[ TableVault Record ]---


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

---[ TableVault Record ]---



In [8]:
vault.create_embedding_list("sentence-transformers-sentence-1-threshold", ndim=384)
vault.create_embedding_list("sentence-transformers-sentence-2-threshold", ndim=384)

for i in range(len(emb1_list)):
    vault.append_embedding("sentence-transformers-sentence-1-threshold", emb1_list[i], 
                       input_items = {"glue_mrpc_validation": [i, i + 1]}
                       )
    vault.append_embedding("sentence-transformers-sentence-2-threshold", emb2_list[i], 
                       input_items = {"glue_mrpc_validation": [i, i + 1]}
                       )


description = "This dataset stores the normalized SentenceTransformer embeddings for the sentence1 field of each example in the GLUE MRPC validation set. Each item is a single 384-dimensional float vector produced by the model sentence-transformers/paraphrase-MiniLM-L3-v2, with one embedding per input row and lineage linked back to the corresponding glue_mrpc_validation record.\n\nStructure: an embedding list with ndim=384. There are no tabular feature columns; each entry consists of the embedding vector plus metadata links to the source row in glue_mrpc_validation.\n\nRole in the workflow: this dataset represents the first sentence in each MRPC pair in embedding space. These vectors are used with the corresponding sentence-2 embeddings to compute cosine similarity scores, which are then thresholded (0.74) to generate paraphrase predictions."
embedding = get_embeddings(description)
vault.create_description("sentence-transformers-sentence-1-threshold", description, embedding)

properties = {"task": "paraphrase detection", "modality": "text", "representation": "sentence embedding", "content": "sentence1 embeddings", "sentence_field": "sentence1", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "sentence-transformers/paraphrase-MiniLM-L3-v2", "embedding_dim": "384", "similarity": "cosine", "normalized": "true", "domain": "news"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence-transformers-sentence-1-threshold", cat, embedding, prop)

description = "sentence-transformers-sentence-2-threshold is an embedding dataset containing one normalized 384-dimensional sentence embedding for the sentence2 field of each example in the GLUE MRPC validation set. Each entry corresponds 1-to-1 with a row in glue_mrpc_validation and is linked back to the source record through input_items, so the dataset structure is an ordered list of embedding vectors plus provenance to the original example index range. The embeddings are produced with the sentence-transformers/paraphrase-MiniLM-L3-v2 model and are intended to represent the second sentence in each MRPC pair. In this workflow, this dataset is used together with the matching sentence-transformers-sentence-1-threshold embeddings to compute cosine similarity scores and generate threshold-based paraphrase predictions."
embedding = get_embeddings(description)
vault.create_description("sentence-transformers-sentence-2-threshold", description, embedding)

properties = {"task": "paraphrase detection", "modality": "text", "representation": "sentence embedding", "embedding_model": "sentence-transformers/paraphrase-MiniLM-L3-v2", "embedding_dim": "384", "normalized": "True", "similarity": "cosine", "input_field": "sentence2", "paired_with": "sentence1", "split": "validation", "size": "408", "source": "glue/mrpc", "language": "en", "domain": "news", "threshold": "0.74"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence-transformers-sentence-2-threshold", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [9]:
scores = (emb1 * emb2).sum(dim=1)
y_pred = (scores >= threshold).long().cpu().numpy()
scores_np = scores.detach().cpu().numpy()

print("done")
print("embedding_shape:", tuple(emb1.shape))
print("score_range:", float(scores_np.min()), float(scores_np.max()))


---[ TableVault Record ]---
done
embedding_shape: (408, 384)
score_range: 0.30519944429397583 0.9930030703544617
---[ TableVault Record ]---



In [10]:
vault.create_record_list("sentence_transformers_mrpc_prediction_threshold", column_names=["prediction"])

for i in range(len(y_pred)):
    vault.append_record("sentence_transformers_mrpc_prediction_threshold", {"prediction": int(y_pred[i])}, 
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                           "sentence-transformers-sentence-1-threshold": [i, i + 1],
                           "sentence-transformers-sentence-2-threshold": [i, i + 1],
                       }
                       )

description = "sentence_transformers_mrpc_prediction_threshold is a record list containing the per-example binary predictions produced on the GLUE MRPC validation set by a SentenceTransformer paraphrase model. Each record has a single field, prediction, where 1 indicates the sentence pair is predicted to be a paraphrase and 0 indicates not_paraphrase. Predictions are generated by encoding sentence1 and sentence2 with sentence-transformers/paraphrase-MiniLM-L3-v2, computing their cosine similarity via normalized embedding dot product, and applying a fixed threshold of 0.74. The dataset is aligned one-to-one with glue_mrpc_validation and is linked to the corresponding sentence-pair embeddings, making it the main stored output used for error analysis, metric computation, and creation of the workflow summary dataset."
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_mrpc_prediction_threshold", description, embedding)

properties = {"dataset_type": "model predictions", "task": "paraphrase detection", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "sentence-transformers/paraphrase-MiniLM-L3-v2", "similarity_metric": "cosine similarity", "threshold": "0.74", "label_type": "binary", "output_column": "prediction", "input_type": "sentence pair", "domain": "newswire"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_mrpc_prediction_threshold", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [11]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


---[ TableVault Record ]---
{'accuracy': 0.7132352941176471, 'f1': 0.7929203539823009}
                precision    recall  f1-score   support

not_paraphrase       0.55      0.52      0.53       129
    paraphrase       0.78      0.80      0.79       279

      accuracy                           0.71       408
     macro avg       0.67      0.66      0.66       408
  weighted avg       0.71      0.71      0.71       408

---[ TableVault Record ]---



In [12]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores_np[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score: 0.9407629370689392
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.30519944429397583
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.8386293649673462
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announ

In [13]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores_np[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


---[ TableVault Record ]---
num_errors: 117
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.8386293649673462
true: 0 pred: 1
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
score: 0.6254448890686035
true: 1 pred: 0
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
score: 0.7619279623031616
true: 0 pred: 1
idx: 11
sentence1: " Sanitati

In [14]:
vault.create_record_list("st_paraphrase_minilm_l3_cosine_threshold_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("st_paraphrase_minilm_l3_cosine_threshold_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "sentence_transformers_mrpc_prediction_threshold": [0, len(ds)]
                    })

summary

description = "This dataset stores the run-level evaluation summary for the SentenceTransformer bi-encoder cosine-similarity paraphrase detection experiment on the GLUE MRPC validation set. It contains a single summary record rather than per-example predictions. The fields are: accuracy (float), f1 (float), and classification_report (string with class-wise precision, recall, and F1 statistics). In this workflow, it serves as the compact results table that aggregates model performance after generating sentence embeddings, scoring sentence pairs with cosine similarity, applying a fixed decision threshold, and comparing predictions against the MRPC ground-truth labels."
embedding = get_embeddings(description)
vault.create_description("st_paraphrase_minilm_l3_cosine_threshold_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "source": "glue/mrpc", "split": "validation", "size": "408", "domain": "news", "model": "sentence-transformers/paraphrase-MiniLM-L3-v2", "method": "bi-encoder cosine similarity thresholding", "similarity_metric": "cosine", "threshold": "0.74", "input_type": "sentence pair", "output_type": "aggregate metrics", "metrics": "accuracy,f1,classification_report", "labels": "not_paraphrase,paraphrase"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("st_paraphrase_minilm_l3_cosine_threshold_mrpc_summary", cat, embedding, prop)



---[ TableVault Record ]---
---[ TableVault Record ]---



In [15]:
description = "This notebook evaluates a sentence-transformer baseline for paraphrase detection on the GLUE MRPC validation set and documents the full workflow in TableVault. It uses the sentence-transformers/paraphrase-MiniLM-L3-v2 model to encode each sentence pair into 384-dimensional normalized embeddings, computes cosine similarity via the embedding dot product, and applies a fixed threshold of 0.74 to predict whether the two sentences are paraphrases. The notebook loads sentence1, sentence2, and gold labels from glue_mrpc_validation, generates embeddings for both sentence columns in batches, and stores those embeddings in TableVault with lineage links back to the source records. It then creates a prediction record list, evaluates performance with accuracy, F1, and a full classification report, and prints example predictions plus a small set of misclassified cases for qualitative inspection. In addition, it saves an experiment summary record and attaches natural-language descriptions and metadata properties to the process outputs using OpenAI text embeddings, so the dataset, intermediate artifacts, predictions, and overall experiment can be discovered and understood later." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("st_paraphrase_minilm_l3_cosine_threshold_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "problem_type": "binary text pair classification", "approach": "sentence embedding similarity thresholding", "model": "sentence-transformers/paraphrase-MiniLM-L3-v2", "model_family": "SentenceTransformer MiniLM bi-encoder", "embedding_model_for_metadata": "text-embedding-3-large", "dataset": "glue/mrpc", "data_split": "validation", "similarity_metric": "cosine similarity", "decision_rule": "score >= threshold", "threshold": "0.74", "embedding_dimension": "384", "frameworks": "sentence-transformers, PyTorch, scikit-learn, Hugging Face Datasets", "evaluation_metrics": "accuracy, f1-score, classification report", "storage_backend": "TableVault with ArangoDB", "hardware": "Apple Metal Performance Shaders (MPS) or CPU", "process_name": "st_paraphrase_minilm_l3_cosine_threshold_mrpc"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("st_paraphrase_minilm_l3_cosine_threshold_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

